# "THE PRICE IS RIGHT" — Week 8, Day 5

## The finale: build the user interface

Day 5 wraps the Week 8 agents in a Gradio dashboard. We will build the interface in small pieces, inspect persistent deal memory, and then launch the complete application.

In [1]:
import logging
import os
import sys
from pathlib import Path

import gradio as gr
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find the repository .env file")
load_dotenv(dotenv_path, override=True)

REPO_ROOT = Path(dotenv_path).parent
WEEK8_DIR = REPO_ROOT / "lectures" / "week-eight"
if str(WEEK8_DIR) not in sys.path:
    sys.path.insert(0, str(WEEK8_DIR))

from agents.deals import Deal, Opportunity
from deal_agent_framework import DealAgentFramework

logging.getLogger().setLevel(logging.INFO)
print(f"Gradio {gr.__version__}")

/Users/marcolerma/GitHub/applied-llm-engineering/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Gradio 4.44.1


## 1. Start with a title

`gr.Blocks` is a container that lets us compose rows, columns, Markdown, tables, plots, and events. Running the next cell starts a local server; stop it before moving to the following UI example.

In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True) as title_demo:
    gr.Markdown(
        '<div style="text-align:center;font-size:24px">'
        '<strong>The Price is Right</strong> — Deal Hunting Agentic AI</div>'
    )
    gr.Markdown(
        '<div style="text-align:center;font-size:14px">'
        'Week 8 agents collaborating to find and price online deals.</div>'
    )

# Uncomment to launch this small demonstration.
#title_demo.launch(inbrowser=True)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## 2. Display opportunities as a table

Gradio state holds Python objects between events. The display function converts each `Opportunity` into ordinary table cells.

In [4]:
example_deal = Deal(
    product_description="Example product description",
    price=100.0,
    url="https://example.com/deal",
)
example_opportunity = Opportunity(
    deal=example_deal,
    estimate=200.0,
    discount=100.0,
)

def table_for(opportunities):
    return [
        [item.deal.product_description, item.deal.price, item.estimate, item.discount, item.deal.url]
        for item in opportunities
    ]

table_for([example_opportunity])

[['Example product description',
  100.0,
  200.0,
  100.0,
  'https://example.com/deal']]

In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True) as table_demo:
    opportunity_state = gr.State([example_opportunity])
    gr.Markdown("## Deals surfaced so far")
    opportunity_table = gr.Dataframe(
        headers=["Description", "Price", "Estimate", "Discount", "URL"],
        wrap=True,
        column_widths=[4, 1, 1, 1, 2],
        row_count=10,
        col_count=5,
        height=400,
        interactive=False,
    )
    table_demo.load(table_for, inputs=[opportunity_state], outputs=[opportunity_table])

# Uncomment to launch this demonstration.
    #table_demo.launch(inbrowser=True)

Matplotlib is building the font cache; this may take a moment.


Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


## 3. Inspect persistent memory

The framework stores surfaced deals in `memory.json`, always resolves paths relative to Week 8, and automatically chooses the populated `products` or `products_lite` Chroma collection. Initializing the framework does **not** call OpenAI, Modal, or Pushover.

In [8]:
framework = DealAgentFramework()
print(f"RAG collection: {framework.collection.name} ({framework.collection.count():,} products)")
print(f"Remembered opportunities: {len(framework.memory)}")
table_for(framework.memory)

[2026-08-19 12:36:49 -0400] [Agents] [INFO] [Agent Framework] Using Chroma collection 'products_lite' with 20,000 products
RAG collection: products_lite (20,000 products)
Remembered opportunities: 0


[]

To clear all saved opportunities, explicitly run:

```python
DealAgentFramework.reset_memory()
```

This is intentionally not executed automatically.

## 4. Run the final application

The standalone app includes:

- a deal table backed by persistent memory;
- live agent logs;
- a 3D t-SNE visualization of product embeddings;
- an automatic scan on load and every five minutes;
- click-to-repeat notifications for saved deals.

Launching it starts live OpenAI, Modal, RSS, and optional Pushover work. Pushover delivery is skipped when its environment variables are absent.

In [ ]:
# Run this cell when you are ready to launch the complete app.
# It blocks while the Gradio server is running; interrupt the cell to stop it.

import subprocess
subprocess.run([sys.executable, str(WEEK8_DIR / "price_is_right.py")], check=True)

/Users/marcolerma/GitHub/applied-llm-engineering/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[2026-08-19 12:37:54 -0400] [Agents] [INFO] [Agent Framework] Using Chroma collection 'products_lite' with 20,000 products
Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Planning Agent] Planning Agent is initializing
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Ensemble Agent] Initializing Ensemble Agent
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Specialist Agent] Specialist Agent is initializing - connecting to modal
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Frontier Agent] Initializing Frontier Agent
[2026-08-19 12:37:57 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent is setting up with OpenAI
[2026-08-19 12:37:57

12:39:16 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
12:39:32 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler


[2026-08-19 12:39:32 -0400] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-08-19 12:39:32 -0400] [Agents] [INFO] [Ensemble Agent] Pre-processed text using openai/gpt-5-nano
[2026-08-19 12:39:32 -0400] [Agents] [INFO] [Specialist Agent] Specialist Agent is calling remote fine-tuned model
[2026-08-19 12:40:15 -0400] [Agents] [INFO] [Specialist Agent] Specialist Agent completed - predicting $25.00
[2026-08-19 12:40:15 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.82s/it]


...

[2026-08-19 12:41:18 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent has found similar products
[2026-08-19 12:41:18 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent completed - predicting $79.99
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is starting a prediction
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Neural Network Agent] Neural Network Agent completed - predicting $44.00
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Ensemble Agent] Ensemble Agent complete - returning $76.49
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Planning Agent] Processed deal with discount $-73.51
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Planning Agent] Pricing a potential deal
[2026-08-19 12:41:25 -0400] [Agents] [INFO] [Ensemble Agent] Running Ensemble Agent - preprocessing text
[2026-08-19 12:41:25 

12:41:25 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
12:41:34 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler


[2026-08-19 12:41:34 -0400] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-08-19 12:41:34 -0400] [Agents] [INFO] [Ensemble Agent] Pre-processed text using openai/gpt-5-nano
[2026-08-19 12:41:34 -0400] [Agents] [INFO] [Specialist Agent] Specialist Agent is calling remote fine-tuned model
[2026-08-19 12:41:34 -0400] [Agents] [INFO] [Specialist Agent] Specialist Agent completed - predicting $500.00
[2026-08-19 12:41:34 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products
[2026-08-19 12:41:35 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent has found similar products
[2026-08-19 12:41:35 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.55it/s]


[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Frontier Agent] Frontier Agent completed - predicting $499.99
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is starting a prediction
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Neural Network Agent] Neural Network Agent completed - predicting $299.12
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Ensemble Agent] Ensemble Agent complete - returning $479.90
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Planning Agent] Processed deal with discount $309.90
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Planning Agent] Best deal has discount $309.90
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Messaging Agent] Messaging Agent is sending a push notification
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Planning Agent] Planning run completed
[2026-08-19 12:41:41 -0400] [Agents] [INFO] [Agent Framework] Planning Agent retu

You can also launch the finale from a terminal:

```bash
cd ~/GitHub/applied-llm-engineering
source .venv/bin/activate
python lectures/week-eight/price_is_right.py
```

The browser should open automatically. Watch the logs while the agents scan, estimate, rank, remember, and optionally notify.